In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

PROJECT_DIR = Path.cwd().parent

# Lokasi dataset asli
PABRIK_DIR = PROJECT_DIR / "datasets" / "dataset-pabrik-kalsel"

KALTENG_DIR = (
    PROJECT_DIR
    / "datasets"
    / "dataset-perkebunan-kalteng"
    / "yolov8"
)

# Lokasi penyimpanan anotasi hasil harmonisasi
OUTPUT_DIR = PROJECT_DIR / "datasets" / "processed"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Pabrik ditemukan:", PABRIK_DIR.exists())
print("Kalteng ditemukan:", KALTENG_DIR.exists())
print("Output:", OUTPUT_DIR)

In [ ]:
def remap_labels(
    dataset_name,
    split_name,
    image_dir,
    label_dir,
    class_mapping
):
    """
    Menghasilkan label YOLO baru berdasarkan class_mapping.
    File asli tidak diubah.
    """

    output_label_dir = (
        OUTPUT_DIR
        / dataset_name
        / "labels"
        / split_name
    )
    output_label_dir.mkdir(parents=True, exist_ok=True)

    total_images = 0
    total_objects_original = 0
    total_objects_kept = Counter()
    images_without_target = 0
    missing_labels = 0

    image_files = [
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for image_path in image_files:
        total_images += 1

        label_path = label_dir / f"{image_path.stem}.txt"

        mapped_lines = []

        if label_path.exists():
            lines = label_path.read_text(
                encoding="utf-8"
            ).splitlines()

            for line in lines:
                if not line.strip():
                    continue

                values = line.split()

                if len(values) != 5:
                    raise ValueError(
                        f"Format label tidak valid: {label_path}"
                    )

                original_class = int(values[0])
                total_objects_original += 1

                if original_class in class_mapping:
                    new_class = class_mapping[original_class]

                    mapped_lines.append(
                        " ".join(
                            [str(new_class)] + values[1:]
                        )
                    )

                    total_objects_kept[new_class] += 1

        else:
            missing_labels += 1

        if not mapped_lines:
            images_without_target += 1

        output_path = output_label_dir / f"{image_path.stem}.txt"

        output_path.write_text(
            "\n".join(mapped_lines),
            encoding="utf-8"
        )

    return {
        "Dataset": dataset_name,
        "Split": split_name,
        "Jumlah Gambar": total_images,
        "Objek Asli": total_objects_original,
        "Unripe (ID 0)": total_objects_kept[0],
        "Ripe (ID 1)": total_objects_kept[1],
        "Gambar Tanpa Target": images_without_target,
        "Label Asli Tidak Ditemukan": missing_labels
    }


# Mapping ID kelas asli menjadi ID kelas target
PABRIK_MAPPING = {
    4: 0,  # TBS mentah -> Unripe
    3: 1   # TBS masak -> Ripe
}

KALTENG_MAPPING = {
    4: 0,  # Unripe -> Unripe
    2: 1   # Ripe -> Ripe
}

results_mapping = []

# Dataset Pabrik Kalsel
pabrik_splits = {
    "train": PABRIK_DIR / "train_rev2" / "train",
    "valid": PABRIK_DIR / "valid_rev2" / "valid",
    "test": PABRIK_DIR / "test_rev2" / "test"
}

for split, folder in pabrik_splits.items():
    results_mapping.append(
        remap_labels(
            "dataset-pabrik-kalsel",
            split,
            folder,
            folder,
            PABRIK_MAPPING
        )
    )


# Dataset Perkebunan Kalteng
kalteng_splits = {
    "train": KALTENG_DIR / "train",
    "valid": KALTENG_DIR / "valid",
    "test": KALTENG_DIR / "test"
}

for split, folder in kalteng_splits.items():
    results_mapping.append(
        remap_labels(
            "dataset-perkebunan-kalteng",
            split,
            folder / "images",
            folder / "labels",
            KALTENG_MAPPING
        )
    )


df_mapping_results = pd.DataFrame(results_mapping)

display(df_mapping_results)

In [ ]:
# Simpan ringkasan hasil harmonisasi

MAPPING_RESULT_DIR = (
    PROJECT_DIR / "results" / "class_mapping"
)

MAPPING_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df_mapping_results.to_csv(
    MAPPING_RESULT_DIR / "mapping_summary.csv",
    index=False
)

print("Ringkasan mapping berhasil disimpan.")
print("Lokasi:", MAPPING_RESULT_DIR)

In [ ]:
# Validasi ID kelas pada hasil mapping

validation_mapping = []

for dataset_name in [
    "dataset-pabrik-kalsel",
    "dataset-perkebunan-kalteng"
]:
    for split in ["train", "valid", "test"]:

        label_dir = (
            OUTPUT_DIR
            / dataset_name
            / "labels"
            / split
        )

        count_images = 0
        count_empty = 0
        class_counts = Counter()
        invalid_ids = 0

        for label_path in label_dir.glob("*.txt"):
            count_images += 1

            lines = [
                line for line in label_path.read_text(
                    encoding="utf-8"
                ).splitlines()
                if line.strip()
            ]

            if not lines:
                count_empty += 1

            for line in lines:
                values = line.split()
                class_id = int(values[0])

                if class_id not in [0, 1]:
                    invalid_ids += 1
                else:
                    class_counts[class_id] += 1

        validation_mapping.append({
            "Dataset": dataset_name,
            "Split": split,
            "Jumlah Label": count_images,
            "Label Kosong": count_empty,
            "Unripe": class_counts[0],
            "Ripe": class_counts[1],
            "Invalid ID": invalid_ids
        })

df_validation_mapping = pd.DataFrame(validation_mapping)

display(df_validation_mapping)